### 1. Setup

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import chi2_contingency

np.random.seed(2026)
sns.set_style("whitegrid")

### 2. Load & select cluster variables

In [37]:
df = pd.read_csv("Citizen_Satisfaction_Survey_2018_2021.csv")

cluster_vars = ["q39","q34","q37","q38","q30","q10","q11a","q12"]

X = (df[cluster_vars]
       .apply(pd.to_numeric, errors="coerce")
       .dropna())

print("Rows kept:", len(X), "of", len(df))
X.describe().round(2)

Rows kept: 10002 of 10002


,q39,q34,q37,q38,q30,q10,q11a,q12
count,10002.00,10002.00,10002.00,10002.00,10002.00,10002.00,10002.00,10002.00
mean,5.98,1.25,6.73,2.35,4.31,3.13,6.47,2.75
std,2.60,0.60,3.16,0.75,1.58,0.86,2.06,1.20
min,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
25%,4.00,1.00,4.00,2.00,3.00,3.00,5.00,2.00
50%,7.00,1.00,7.00,2.00,5.00,3.00,7.00,3.00
75%,8.00,1.00,10.00,3.00,6.00,4.00,8.00,4.00
max,9.00,5.00,12.00,4.00,7.00,5.00,11.00,6.00


### 3. Standardize

In [38]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=cluster_vars, index=X.index)

### 4. Choose k — elbow + silhouette

In [ ]:
inertia, sil = [], []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=25, random_state=42).fit(X_scaled)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(k_range, inertia, marker="o")
ax[0].set(title="Elbow — inertia", xlabel="k", ylabel="Inertia")
ax[1].plot(k_range, sil, marker="o", color="green")
ax[1].set(title="Silhouette", xlabel="k", ylabel="Score")
plt.tight_layout(); plt.show()

pd.DataFrame({"k": list(k_range), "inertia": inertia, "silhouette": sil}).round(3)


### 5. K-means

In [ ]:
K = 4   # Based on the Elbow abd silhouette plots above

km = KMeans(n_clusters=K, n_init=25, random_state=42).fit(X_scaled)
df.loc[X.index, "cluster"] = km.labels_
df["cluster"] = df["cluster"].astype("Int64")

### 6. PCA projection

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=coords[:,0], y=coords[:,1], hue=km.labels_, palette="viridis", s=20)
plt.title(f"K-means clusters (k={K}) — PCA projection")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.legend(title="Cluster")
plt.tight_layout(); plt.show()

### 7. Cluster profile table

In [ ]:
profile = (
    df.loc[X.index]
      .groupby("cluster")
      .agg(
          n=(cluster_vars[0], "size"),
          **{v: (v, "mean") for v in cluster_vars}
      )
)
profile["pct"] = (profile["n"] / profile["n"].sum() * 100).round(1)
profile.round(2)

8. Demographic profiling (χ² + Cramér's V)

In [ ]:
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    return np.sqrt(chi2 / (n * (min(ct.shape) - 1))), chi2, ct.shape

for var in ["s4qt", "market2", "sexfix", "qwave"]:
    v, chi2, shape = cramers_v(df.loc[X.index, "cluster"], df.loc[X.index, var])
    print(f"{var:10s}  χ²={chi2:10.2f}  Cramér's V={v:.3f}")

# Cross-tab example
ct = pd.crosstab(df.loc[X.index,"cluster"], df.loc[X.index,"s4qt"], normalize="index")
print(ct.round(2))

### 9. Silhouette plot

In [ ]:
sample_sil = silhouette_samples(X_scaled, km.labels_)

fig, ax = plt.subplots(figsize=(7, 5))
y_lower = 10
for c in range(K):
    vals = np.sort(sample_sil[km.labels_ == c])
    size = len(vals)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=0.7)
    ax.text(-0.05, y_lower + size/2, str(c))
    y_lower = y_upper + 10

ax.axvline(silhouette_score(X_scaled, km.labels_), color="red", ls="--")
ax.set(title="Silhouette per cluster", xlabel="Silhouette", ylabel="Cluster")
plt.tight_layout(); plt.show()

### 10. Hierarchical (Ward) alternative

In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(10, 4))
dendrogram(Z, truncate_mode="lastp", p=20, no_labels=True)
plt.title("Ward dendrogram (truncated)")
plt.tight_layout(); plt.show()

hc = AgglomerativeClustering(n_clusters=K, linkage="ward").fit(X_scaled)
df.loc[X.index, "cluster_h"] = hc.labels_

# Agreement with k-means
agree = (df.loc[X.index, "cluster"] == df.loc[X.index, "cluster_h"]).mean()
print(f"Agreement k-means vs Ward: {agree:.1%}")

### 11. Latent Class Analysis (categorical items)


In [ ]:
# pip install stepmix
from stepmix.stepmix import StepMix

LCA_vars = [f"q8_{i}" for i in range(1, 6)]   # pick items
LCA_data = df[LCA_vars].apply(pd.to_numeric, errors="coerce").dropna()

lca = StepMix(n_components=4, measurement="categorical",
              n_init=10, random_state=42)
lca.fit(LCA_data)

print("Class sizes:", np.round(lca.predict_proba(LCA_data).mean(0), 3))
print("BIC:", lca.bic(LCA_data))

### 12. Validation checklist

In [ ]:
print("Silhouette (global):", round(silhouette_score(X_scaled, km.labels_), 3))
print("Cluster sizes:\n", pd.Series(km.labels_).value_counts().sort_index())

# Split-half stability
from sklearn.model_selection import train_test_split
idx1, idx2 = train_test_split(X_scaled.index, test_size=0.5, random_state=42)

k1 = KMeans(K, n_init=25, random_state=42).fit(X_scaled.loc[idx1])
k2 = KMeans(K, n_init=25, random_state=42).fit(X_scaled.loc[idx2])

common = idx1.intersection(idx2)
if len(common):
    agree = (k1.predict(X_scaled.loc[common]) == k2.predict(X_scaled.loc[common])).mean()
    print(f"Split-half agreement: {agree:.1%}")

### 13. APA-ready output

In [ ]:
def apa_profile_table(profile, cluster_vars):
    out = profile[["n","pct"] + cluster_vars].round(2)
    out.index = [f"Cluster {i}" for i in out.index]
    return out

print(apa_profile_table(profile, cluster_vars))

A k-means cluster analysis was conducted on eight satisfaction and trust items (q39, q34, q37, q38, q30, q10, q11a, q12). Variables were standardized (*z*-scores) prior to clustering. The optimal number of clusters was determined using the elbow method and average silhouette width, supporting a four-cluster solution (silhouette = [value]). Cluster sizes ranged from [min] to [max] respondents ([min%]–[max%]). A χ² test indicated cluster membership was significantly associated with quadrant, χ²([df], N = [N]) = [χ²], *p* < .001, Cramér's V = [V]. Cluster [x] reported the highest overall satisfaction (M = [m], SD = [sd]), whereas Cluster [y] reported the lowest (M = [m], SD = [sd]). Split-half validation showed [%] agreement, supporting solution stability.